# Shinobu L4 GPU 声线后端（IndexTTS-2.5 · OpenAI 兼容）

专为 **Google Colab L4 GPU**（兼容 T4 / A100）打造，配合本机 `127.0.0.1:18791` 双模语音网关直连 **Project AIRI** 或任意 OpenAI 兼容社区外壳。

### 使用步骤（首次约 3~5 分钟，后续走 Drive 缓存约 1 分钟）
1. 顶部菜单 **代码执行程序 → 更改运行时类型 → 选择 L4 GPU**。
2. 准备一段 **5～15 秒无背景音乐的干净参考干音**，直接点「全部运行」（Cell 3 会引导上传并自动存到 Google Drive 复用）。
3. Cell 4 跑通后会打印 **Colab URL** 和 **API Key**；回到本机终端执行 `shinobu tts-gateway --set-upstream <Colab URL> --api-key <API Key>`，AIRI 无需改任何配置即刻秒切 L4 满血声线！


In [ ]:
#@title 1. 检查 L4 GPU、可选挂载 Google Drive 并安装 IndexTTS-2.5（uv 隔离环境）
USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/shinobu_tts"  #@param {type:"string"}

from pathlib import Path

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
    print(f"[OK] Google Drive cache dir: {DRIVE_ROOT}")

!pip install -q -U uv "huggingface-hub[cli]"

if not Path("/content/index-tts").exists():
    !git clone --depth 1 https://github.com/index-tts/index-tts.git /content/index-tts

%cd /content/index-tts
!uv sync --all-extras

if not Path("/usr/local/bin/cloudflared").exists():
    !curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

print("[OK] Environment ready!")


In [ ]:
#@title 2. 下载或复用 IndexTTS-2.5 模型权重（优先走 Drive 缓存）
import shutil
from pathlib import Path

ckpt_dir = Path(DRIVE_ROOT) / "checkpoints_v25" if USE_DRIVE else Path("/content/index-tts/checkpoints")
ckpt_dir.mkdir(parents=True, exist_ok=True)

if not (ckpt_dir / "config.yaml").is_file():
    print(f"[Downloading] IndexTeam/IndexTTS-2.5 -> {ckpt_dir} ...")
    !hf download IndexTeam/IndexTTS-2.5 --local-dir="{ckpt_dir}"
else:
    print(f"[Cache Hit] Reusing weights from {ckpt_dir}")

local_ckpt = Path("/content/index-tts/checkpoints")
if str(ckpt_dir.resolve()) != str(local_ckpt.resolve()):
    if local_ckpt.is_symlink() or local_ckpt.is_file():
        local_ckpt.unlink()
    elif local_ckpt.is_dir():
        shutil.rmtree(local_ckpt)
    local_ckpt.symlink_to(ckpt_dir)

print("[OK] Checkpoints ready:", sorted(p.name for p in local_ckpt.iterdir())[:10])


In [ ]:
#@title 3. 准备参考干音（5~15 秒干净人声，优先复用 Drive）
import subprocess
from pathlib import Path

ref_wav = Path(DRIVE_ROOT) / "ref.wav" if USE_DRIVE else Path("/content/ref.wav")
FORCE_REUPLOAD = False  #@param {type:"boolean"}

if ref_wav.is_file() and not FORCE_REUPLOAD:
    print(f"[OK] Using cached reference voice: {ref_wav} ({ref_wav.stat().st_size} bytes)")
else:
    from google.colab import files
    print("请上传一段 5~15 秒无背景音乐的干净人声参考音频...")
    uploaded = files.upload()
    assert uploaded, "未选择参考音频文件！"
    src_name = next(iter(uploaded.keys()))
    src_path = Path("/content") / src_name
    if not src_path.exists():
        src_path = Path.cwd() / src_name
    subprocess.run(
        ["ffmpeg", "-y", "-i", str(src_path), "-ac", "1", "-ar", "24000", str(ref_wav)],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    print(f"[OK] Normalized and saved reference voice -> {ref_wav}")


In [ ]:
#@title 4. 启动 L4 bf16 OpenAI /v1/audio/speech 服务 + Cloudflare 加密隧道
import os, re, secrets, subprocess, time
from pathlib import Path

API_KEY = "shinobu-l4-" + secrets.token_urlsafe(12)
REF_WAV = str(Path(DRIVE_ROOT) / "ref.wav" if USE_DRIVE else Path("/content/ref.wav"))
assert Path(REF_WAV).is_file(), f"找不到参考干音 {REF_WAV}，请先运行 Cell 3！"

server_code = r'''
import json, os, re, sys, tempfile, threading, time
from http import HTTPStatus
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from indextts.infer_v2_5 import IndexTTS2

API_KEY = os.environ.get("SHINOBU_TTS_API_KEY", "").strip()
REF_WAV = os.environ["SHINOBU_REF_WAV"]
PORT = int(os.environ.get("SHINOBU_TTS_PORT", "8080"))

_JA_RE = re.compile(r"[\u3040-\u30ff]")
_ZH_RE = re.compile(r"[\u4e00-\u9fff]")

def detect_lang(text: str) -> str:
    if _JA_RE.search(text):
        return "JA"
    if _ZH_RE.search(text):
        return "ZH"
    return "EN"

print("[IndexTTS-2.5] Loading model on CUDA (use_bf16=True)...", flush=True)
tts = IndexTTS2(
    cfg_path="checkpoints/config.yaml",
    model_dir="checkpoints",
    use_bf16=True,
    use_qwen_emo=False,
)
infer_lock = threading.Lock()
print("[IndexTTS-2.5] Model loaded! Warming up...", flush=True)
with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tf:
    warm_path = tf.name
try:
    tts.infer(spk_audio_prompt=REF_WAV, text="吾乃怪异之王。", output_path=warm_path, lang="ZH")
    print("[IndexTTS-2.5] Warmup done!", flush=True)
finally:
    if os.path.exists(warm_path):
        os.remove(warm_path)

class Handler(BaseHTTPRequestHandler):
    def log_message(self, fmt, *args):
        sys.stdout.write(f"[L4-TTS] {fmt % args}\n")
        sys.stdout.flush()

    def _cors(self):
        self.send_header("Access-Control-Allow-Origin", "*")
        self.send_header("Access-Control-Allow-Methods", "GET, POST, OPTIONS")
        self.send_header("Access-Control-Allow-Headers", "Authorization, Content-Type")

    def _json(self, status, obj):
        raw = json.dumps(obj, ensure_ascii=False).encode("utf-8")
        self.send_response(status)
        self._cors()
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(raw)))
        self.end_headers()
        self.wfile.write(raw)

    def _check_auth(self) -> bool:
        if not API_KEY:
            return True
        return self.headers.get("Authorization", "") == f"Bearer {API_KEY}"

    def do_OPTIONS(self):
        self.send_response(HTTPStatus.NO_CONTENT)
        self._cors()
        self.send_header("Content-Length", "0")
        self.end_headers()

    def do_GET(self):
        path = self.path.split("?", 1)[0].rstrip("/") or "/"
        if path in ("/", "/health"):
            self._json(200, {"status": "ok", "model": "IndexTTS-2.5", "device": "cuda-bf16"})
            return
        if path in ("/v1/models", "/models"):
            self._json(200, {"object": "list", "data": [{"id": "indextts-2.5", "object": "model"}, {"id": "tts-1", "object": "model"}]})
            return
        self._json(404, {"error": {"message": f"Not found: {path}"}})

    def do_POST(self):
        path = self.path.split("?", 1)[0].rstrip("/")
        if path not in ("/v1/audio/speech", "/audio/speech"):
            self._json(404, {"error": {"message": f"Not found: {path}"}})
            return
        if not self._check_auth():
            self._json(401, {"error": {"message": "Invalid Bearer token"}})
            return
        n = int(self.headers.get("Content-Length", "0") or "0")
        body = json.loads(self.rfile.read(n).decode("utf-8"))
        text = str(body.get("input") or "").strip()
        if not text:
            self._json(400, {"error": {"message": "Empty input"}})
            return
        lang = str(body.get("lang") or "").strip().upper() or detect_lang(text)
        try:
            speed = max(0.6, min(1.6, float(body.get("speed", 1.0))))
        except Exception:
            speed = 1.0
        duration_factor = round(1.0 / speed, 3)
        t0 = time.time()
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tf:
            out_path = tf.name
        try:
            with infer_lock:
                tts.infer(
                    spk_audio_prompt=REF_WAV,
                    text=text,
                    output_path=out_path,
                    lang=lang,
                    duration_factor=duration_factor,
                )
            audio = Path(out_path).read_bytes()
        finally:
            if os.path.exists(out_path):
                os.remove(out_path)
        dt = time.time() - t0
        self.log_message("synth lang=%s chars=%d bytes=%d elapsed=%.2fs", lang, len(text), len(audio), dt)
        self.send_response(200)
        self._cors()
        self.send_header("Content-Type", "audio/wav")
        self.send_header("Content-Length", str(len(audio)))
        self.end_headers()
        self.wfile.write(audio)

srv = ThreadingHTTPServer(("127.0.0.1", PORT), Handler)
print(f"[L4-TTS] Ready on http://127.0.0.1:{PORT}/v1/audio/speech", flush=True)
srv.serve_forever()
'''

Path("/content/index-tts/colab_tts_server.py").write_text(server_code.strip() + "\n", encoding="utf-8")

env = os.environ.copy()
env["SHINOBU_TTS_API_KEY"] = API_KEY
env["SHINOBU_REF_WAV"] = REF_WAV
env["SHINOBU_TTS_PORT"] = "8080"

srv_proc = subprocess.Popen(
    ["uv", "run", "python", "colab_tts_server.py"],
    cwd="/content/index-tts",
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in srv_proc.stdout:
    print(line, end="")
    if "Ready on http://127.0.0.1:8080" in line:
        break

cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8080"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

tunnel_url = None
t_start = time.time()
for line in cf_proc.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        tunnel_url = m.group(0)
        break
    if time.time() - t_start > 45:
        break

assert tunnel_url, "Cloudflare 隧道拉起超时，请重试本单元格。"

print("\n" + "=" * 66)
print("  SHINOBU L4 TTS IS LIVE! 在本机终端执行以下命令热切上游:")
print(f"  shinobu tts-gateway --set-upstream {tunnel_url}/v1 --api-key {API_KEY}")
print(f"  Colab URL : {tunnel_url}/v1")
print(f"  API Key   : {API_KEY}")
print("=" * 66 + "\n", flush=True)

try:
    for line in srv_proc.stdout:
        print(line, end="")
except KeyboardInterrupt:
    print("Stopping server & tunnel...")
finally:
    cf_proc.terminate()
    srv_proc.terminate()
